In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/python-functions-with-docstrings/python_functions_and_documentation_dataset.csv


In [4]:
# ===== Cell 1: Imports & Config =====
import os, re, json
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
from typing import List
from tqdm import tqdm
tqdm.pandas()

# Paths
DATA_CSV = "/kaggle/input/python-functions-with-docstrings/python_functions_and_documentation_dataset.csv"
OUT_DIR = "/kaggle/working"
BPE_OUT_DIR = os.path.join(OUT_DIR, "bpe")
os.makedirs(BPE_OUT_DIR, exist_ok=True)

# Special tokens
PAD, UNK, BOS, EOS = "<PAD>", "<UNK>", "<BOS>", "<EOS>"


In [5]:
# ===== Cell 2: Load Dataset & Basic EDA =====
print("Loading dataset...")
df = pd.read_csv(DATA_CSV, low_memory=False)
print("Loaded. Shape:", df.shape)

print("\nPartition counts:")
print(df['partition'].value_counts())

print("\nNull counts (code, docstring, summary):")
print(df[['code','docstring','summary']].isnull().sum())

# Clean & add features
df['code_raw'] = df['code'].fillna('').astype(str)
df['docstring_raw'] = df['docstring'].fillna('').astype(str)
df['summary_raw'] = df['summary'].fillna('').astype(str)

print(f"\nEmpty docstrings: {df['docstring_raw'].str.strip().eq('').mean():.2%}")
print(f"Empty summaries: {df['summary_raw'].str.strip().eq('').mean():.2%}")

print("\nTop 5 repos:")
print(df['repo'].value_counts().head())
print("Unique function names:", df['func_name'].nunique())


Loading dataset...
Loaded. Shape: (455243, 13)

Partition counts:
partition
train    410175
valid     22977
test      22091
Name: count, dtype: int64

Null counts (code, docstring, summary):
code         0
docstring    0
summary      0
dtype: int64

Empty docstrings: 0.00%
Empty summaries: 0.00%

Top 5 repos:
repo
saltstack/salt               11111
mitsei/dlkit                  2551
materialsproject/pymatgen     2196
brocade/pynos                 1918
google/grr                    1903
Name: count, dtype: int64
Unique function names: 404597


In [6]:
# ===== Cell 3: Pre-tokenizers =====

# Natural language pretokenizer
_WORD_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)
def pretokenize_text(s: str) -> List[str]:
    return _WORD_RE.findall(s)

# Python code pretokenizer
_CODE_TOKEN_RE = re.compile(
    r"""
    (?:[A-Za-z_][A-Za-z_0-9]*)          # identifiers
    |(?:\d+\.\d+|\d+)                   # numbers
    |(?:"[^"\\]*(?:\\.[^"\\]*)*"        # strings
      |'[^'\\]*(?:\\.[^'\\]*)*')
    |(?:==|!=|<=|>=|->|\+=|\-=|\*=|/=|%=|\*\*|//) # multi-char ops
    |(?:[^\s\w])                        # single-char ops/punct
    """,
    re.VERBOSE
)
def pretokenize_code(s: str) -> List[str]:
    return _CODE_TOKEN_RE.findall(s)


In [7]:
# ===== Cell 4: BPE Tokenizer =====
class BPETokenizer:
    def __init__(self, num_merges=10000, lowercase=False, min_pair_freq=2,
                 specials=(PAD, UNK, BOS, EOS)):
        self.num_merges = num_merges
        self.lowercase = lowercase
        self.min_pair_freq = min_pair_freq
        self.specials = list(specials)
        self.merges, self.merge_ranks, self.vocab = [], {}, {}

    @staticmethod
    def _word_to_symbols(word: str) -> List[str]:
        return list(word) + ["</w>"]

    def _build_initial_vocab(self, lines, pretokenize):
        vocab = Counter()
        for line in lines:
            if self.lowercase: line = line.lower()
            for word in pretokenize(line):
                if not word: continue
                symbols = self._word_to_symbols(word)
                key = " ".join(symbols)
                vocab[key] += 1
        return vocab

    @staticmethod
    def _get_pair_stats(vocab: Counter) -> Counter:
        stats = Counter()
        for word_key, freq in vocab.items():
            symbols = word_key.split()
            for i in range(len(symbols)-1):
                stats[(symbols[i], symbols[i+1])] += freq
        return stats

    @staticmethod
    def _merge_vocab_once(pair, vocab_in: Counter) -> Counter:
        p = re.compile(rf"(?<!\S){re.escape(pair[0])}\s{re.escape(pair[1])}(?!\S)")
        vocab_out = Counter()
        for word_key, freq in vocab_in.items():
            merged = p.sub(pair[0]+pair[1], word_key)
            vocab_out[merged] += freq
        return vocab_out

    def train(self, lines, pretokenize, progress_every=500):
        vocab = self._build_initial_vocab(lines, pretokenize)
        for it in range(self.num_merges):
            stats = self._get_pair_stats(vocab)
            if not stats: break
            best_pair, freq = stats.most_common(1)[0]
            if freq < self.min_pair_freq: break
            vocab = self._merge_vocab_once(best_pair, vocab)
            self.merges.append(best_pair)
            if (it+1) % progress_every == 0:
                print(f"[BPE] {it+1} merges, pair={best_pair}, freq={freq}")
        self.merge_ranks = {p: i for i,p in enumerate(self.merges)}
        symbols = set()
        for k in vocab.keys(): symbols.update(k.split())
        all_tokens = self.specials + sorted(symbols - set(self.specials))
        self.vocab = {tok:i for i,tok in enumerate(all_tokens)}

    def _apply_bpe_to_word(self, word):
        symbols = self._word_to_symbols(word)
        def get_pairs(s): return {(s[i],s[i+1]) for i in range(len(s)-1)}
        pairs = get_pairs(symbols)
        while True:
            mergeable = [(self.merge_ranks[p],p) for p in pairs if p in self.merge_ranks]
            if not mergeable: break
            _, best = min(mergeable)
            new_symbols=[]; i=0
            while i<len(symbols):
                if i<len(symbols)-1 and (symbols[i],symbols[i+1])==best:
                    new_symbols.append(symbols[i]+symbols[i+1]); i+=2
                else:
                    new_symbols.append(symbols[i]); i+=1
            symbols=new_symbols
            if len(symbols)==1: break
            pairs=get_pairs(symbols)
        return symbols

    def encode(self, text, pretokenize):
        if self.lowercase: text = text.lower()
        out=[]
        for word in pretokenize(text):
            out.extend([sub if sub in self.vocab else UNK for sub in self._apply_bpe_to_word(word)])
        return out

    def decode(self, tokens: List[str]) -> str:
        return "".join(tokens).replace("</w>"," ").strip()

    def save(self, dir_path, name):
        os.makedirs(dir_path, exist_ok=True)
        with open(os.path.join(dir_path,f"{name}.merges"),"w") as f:
            for a,b in self.merges: f.write(f"{a}\t{b}\n")
        with open(os.path.join(dir_path,f"{name}.vocab.json"),"w") as f:
            json.dump(self.vocab,f)
        meta={"num_merges":self.num_merges,"lowercase":self.lowercase,
              "min_pair_freq":self.min_pair_freq,"specials":self.specials}
        with open(os.path.join(dir_path,f"{name}.meta.json"),"w") as f:
            json.dump(meta,f)


In [8]:
# ===== Cell 5: Train Tokenizers =====
SAMPLE_N = 5000
NUM_MERGES = 300
MIN_FREQ = 5

def iter_series(series, n=None):
    for i, s in enumerate(series):
        yield str(s)
        if n and i+1>=n: break

print("Training CODE...")
bpe_code = BPETokenizer(NUM_MERGES, lowercase=False, min_pair_freq=MIN_FREQ)
bpe_code.train(iter_series(df['code_raw'], SAMPLE_N), pretokenize_code)
bpe_code.save(BPE_OUT_DIR, "bpe_code")

print("Training DOC...")
bpe_doc = BPETokenizer(NUM_MERGES, lowercase=True, min_pair_freq=MIN_FREQ)
bpe_doc.train(iter_series(df['docstring_raw'], SAMPLE_N), pretokenize_text)
bpe_doc.save(BPE_OUT_DIR, "bpe_doc")

print("Training JOINT...")
def joint_gen(n=None):
    for i,(c,d) in enumerate(zip(df['code_raw'], df['docstring_raw'])):
        yield str(c); yield str(d).lower()
        if n and i+1>=n: break

bpe_joint = BPETokenizer(NUM_MERGES, lowercase=False, min_pair_freq=MIN_FREQ)
bpe_joint.train(joint_gen(SAMPLE_N), pretokenize_code)
bpe_joint.save(BPE_OUT_DIR, "bpe_joint")


Training CODE...
Training DOC...
Training JOINT...


In [11]:
# ===== Cell 6: Evaluation Metrics =====
def jaccard_vocab_overlap(bpe_vocab, ref_tokens):
    flat_ref=set(tok for seq in ref_tokens for tok in seq)
    return len(set(bpe_vocab)&flat_ref)/len(set(bpe_vocab)|flat_ref)

def compression_ratio(texts, encs):
    return float(np.mean([len(t)/len(e) for t,e in zip(texts,encs) if e]))



def consistency_metric(encoded):
    forms=defaultdict(set)
    for seq in encoded:
        word="".join(seq).replace("</w>","")
        forms[word].add(tuple(seq))
    return sum(1 for f in forms.values() if len(f)==1)/len(forms)

def oov_rate(encoded):
    total=sum(len(seq) for seq in encoded)
    unk=sum(tok==UNK for seq in encoded for tok in seq)
    return unk/total if total else 0


In [12]:
def boundary_f1(encoded_texts, ref_tokens_list):
    """
    Compute boundary precision, recall, F1 between BPE encodings and reference tokens.
    encoded_texts: list of lists (BPE tokens)
    ref_tokens_list: list of lists (reference tokens from dataset)
    """
    def get_boundaries(tokens):
        idx = 0
        boundaries = []
        for t in tokens:
            # count characters (ignore </w>)
            idx += len(t.replace("</w>", ""))
            boundaries.append(idx)
        return set(boundaries[:-1])  # exclude final end

    tp = fp = fn = 0
    for enc, ref in zip(encoded_texts, ref_tokens_list):
        b_enc, b_ref = get_boundaries(enc), get_boundaries(ref)
        tp += len(b_enc & b_ref)
        fp += len(b_enc - b_ref)
        fn += len(b_ref - b_enc)

    precision = tp / (tp + fp + 1e-9)
    recall    = tp / (tp + fn + 1e-9)
    f1        = 2 * precision * recall / (precision + recall + 1e-9)
    return precision, recall, f1


In [13]:
enc_code = [bpe_code.encode(c, pretokenize_code) for c in valid_df['code_raw']]
enc_doc  = [bpe_doc.encode(d, pretokenize_text) for d in valid_df['docstring_raw']]

metrics = {
    "jaccard_code": jaccard_vocab_overlap(bpe_code.vocab, valid_df['code_tokens']),
    "jaccard_doc": jaccard_vocab_overlap(bpe_doc.vocab, valid_df['docstring_tokens']),
    "compression_code": compression_ratio(valid_df['code_raw'], enc_code),
    "compression_doc": compression_ratio(valid_df['docstring_raw'], enc_doc),
    "boundary_code": boundary_f1(enc_code, valid_df['code_tokens']),
    "boundary_doc": boundary_f1(enc_doc, valid_df['docstring_tokens']),
    "consistency_code": consistency_metric(enc_code),
    "consistency_doc": consistency_metric(enc_doc),
    "oov_code": oov_rate(enc_code),
    "oov_doc": oov_rate(enc_doc),
}
print(json.dumps(metrics, indent=2))


{
  "jaccard_code": 0.25824175824175827,
  "jaccard_doc": 0.1079136690647482,
  "compression_code": 1.7990124944055805,
  "compression_doc": 2.280646067265021,
  "boundary_code": [
    0.6552579957955674,
    0.4113884658677066,
    0.5054450393334963
  ],
  "boundary_doc": [
    0.3165829145728542,
    0.40210586226520845,
    0.3542558330344042
  ],
  "consistency_code": 1.0,
  "consistency_doc": 1.0,
  "oov_code": 0.20336315043835934,
  "oov_doc": 0.002124468882779305
}


In [18]:
bpe_joint.save(BPE_OUT_DIR, "bpe_joint")

with open(os.path.join(OUT_DIR, "bpe_eval.json"), "w") as f:
    json.dump(metrics, f, indent=2)
